## Libraries Installation

In [1]:
!pip install azure-storage-blob
!pip install snowflake-connector-python
!pip install snowflake-sqlalchemy

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip available: 22.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip available: 22.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip available: 22.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Libraries import

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from bs4 import BeautifulSoup
import os
import re
import requests
from azure.storage.blob import BlobServiceClient
import json
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import io
import warnings
warnings.filterwarnings('ignore')
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine

## Function

In [3]:
def contains_string(r,s):
    return r in s.lower()

## Connection String Load

In [63]:
def load_config_azure(config_path="config.json"):
    """Load the Azure configuration parameters from the config.json file."""
    with open(config_path, "r", encoding="utf-8") as config_file:
        config = json.load(config_file)
    return config["AZURE_CONNECTION_STRING"], config["CONTAINER_NAME"]


def load_config_snowflake(config_path="config.json"):
    """Load the Azure configuration parameters from the config.json file."""
    with open(config_path, "r", encoding="utf-8") as config_file:
        config = json.load(config_file)
    return config["SNOWFLAKE_USER"], config["SNOWFLAKE_PASSWORD"], config["SNOWFLAKE_ACCOUNT"], config["SNOWFLAKE_WAREHOUSE"], config["SNOWFLAKE_DATABASE"], config["SNOWFLAKE_SCHEMA"]

AZURE_CONNECTION_STRING, CONTAINER_NAME = load_config_azure()
SNOWFLAKE_USER, SNOWFLAKE_PASSWORD, SNOWFLAKE_ACCOUNT, SNOWFLAKE_WAREHOUSE, SNOWFLAKE_DATABASE, SNOWFLAKE_SCHEMA = load_config_snowflake()

blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

# Define your Snowflake connection details
connection_params = {
    "user": SNOWFLAKE_USER,
    "password": SNOWFLAKE_PASSWORD,
    "account": SNOWFLAKE_ACCOUNT,
    "warehouse": SNOWFLAKE_WAREHOUSE,
    "database": SNOWFLAKE_DATABASE,
    "schema": SNOWFLAKE_SCHEMA,
}

# Create a Snowflake engine
engine = create_engine(URL(
    user=SNOWFLAKE_USER,
    password= SNOWFLAKE_PASSWORD,
    account=SNOWFLAKE_ACCOUNT,
    warehouse=SNOWFLAKE_WAREHOUSE,
    database=SNOWFLAKE_DATABASE,
    schema=SNOWFLAKE_SCHEMA
))

# Establish connection
# conn = snowflake.connector.connect(**connection_params)

## Extraction

### Extraction Location Data

In [80]:
lookup_blob = "references/lookup.csv"


# Get a BlobClient for the CSV file
blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob=lookup_blob)

# Download the blob content as text
blob_data = blob_client.download_blob().readall()

# Read the CSV data into a pandas DataFrame
lookup_df = pd.read_csv(io.BytesIO(blob_data))

lookup_df.head() # To see the output, run the code.

,Unnamed: 0,OBJECTID,Shape_Leng,Shape_Area,zone,LocationID,borough,latitude,longitude,zipcode
0,0,1,0.116357,0.000782,Newark Airport,1,EWR,40.689488,-74.171526,07114
1,1,2,0.433470,0.004866,Jamaica Bay,2,Queens,40.610791,-73.822490,11693
2,2,3,0.084341,0.000314,Allerton/Pelham Gardens,3,Bronx,40.865745,-73.844947,10469
3,3,4,0.043567,0.000112,Alphabet City,4,Manhattan,40.724137,-73.977726,10009
4,4,5,0.092146,0.000498,Arden Heights,5,Staten Island,40.550665,-74.187537,10312


### Extraction Green Taxi

In [81]:
all_green_dfs = []
for blob in container_client.list_blobs():
  if contains_string("raw/green_tripdata_2024-01.parquet",blob.name) or (contains_string("raw/green_tripdata_2025-01.parquet",blob.name)):
    print(blob.name)
    blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob=blob.name)
    stream = blob_client.download_blob().readall()
    stream = io.BytesIO(stream)
    df = pd.read_parquet(stream)
    all_green_dfs.append(df)
green_dfs = pd.concat(all_green_dfs, ignore_index=True)
print(green_dfs.shape)
green_dfs.head()

raw/green_tripdata_2024-01.parquet
raw/green_tripdata_2025-01.parquet
(104877, 21)


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2024-01-01 00:46:55,2024-01-01 00:58:25,N,1.0,236,239,1.0,1.98,12.8,...,0.5,3.61,0.0,NaN,1.0,21.66,1.0,1.0,2.75,NaN
1,2,2024-01-01 00:31:42,2024-01-01 00:52:34,N,1.0,65,170,5.0,6.54,30.3,...,0.5,7.11,0.0,NaN,1.0,42.66,1.0,1.0,2.75,NaN
2,2,2024-01-01 00:30:21,2024-01-01 00:49:23,N,1.0,74,262,1.0,3.08,19.8,...,0.5,3.00,0.0,NaN,1.0,28.05,1.0,1.0,2.75,NaN
3,1,2024-01-01 00:30:20,2024-01-01 00:42:12,N,1.0,74,116,1.0,2.40,14.2,...,1.5,0.00,0.0,NaN,1.0,16.70,2.0,1.0,0.00,NaN
4,2,2024-01-01 00:32:38,2024-01-01 00:43:37,N,1.0,74,243,1.0,5.14,22.6,...,0.5,6.28,0.0,NaN,1.0,31.38,1.0,1.0,0.00,NaN


## Transformation

### Data Profiling

In [82]:
print(lookup_df.shape)
print(lookup_df.info())

(271, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271 entries, 0 to 270
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  271 non-null    int64  
 1   OBJECTID    271 non-null    int64  
 2   Shape_Leng  271 non-null    float64
 3   Shape_Area  271 non-null    float64
 4   zone        271 non-null    object 
 5   LocationID  271 non-null    int64  
 6   borough     271 non-null    object 
 7   latitude    271 non-null    float64
 8   longitude   271 non-null    float64
 9   zipcode     271 non-null    object 
dtypes: float64(4), int64(3), object(3)
memory usage: 21.3+ KB
None


In [83]:
print(green_dfs.info())
print(green_dfs.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104877 entries, 0 to 104876
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   VendorID               104877 non-null  int32         
 1   lpep_pickup_datetime   104877 non-null  datetime64[us]
 2   lpep_dropoff_datetime  104877 non-null  datetime64[us]
 3   store_and_fwd_flag     99626 non-null   object        
 4   RatecodeID             99626 non-null   float64       
 5   PULocationID           104877 non-null  int32         
 6   DOLocationID           104877 non-null  int32         
 7   passenger_count        99626 non-null   float64       
 8   trip_distance          104877 non-null  float64       
 9   fare_amount            104877 non-null  float64       
 10  extra                  104877 non-null  float64       
 11  mta_tax                104877 non-null  float64       
 12  tip_amount             104877 non-null  floa

### Data Cleaning

In [84]:
lookup_df.drop(['Unnamed: 0', 'OBJECTID', 'Shape_Leng', 'Shape_Area'], axis=1, inplace=True)
# drop row where zipcode is 	07302
lookup_df = lookup_df[lookup_df['zipcode'] != '07302']
lookup_df.head()

,zone,LocationID,borough,latitude,longitude,zipcode
0,Newark Airport,1,EWR,40.689488,-74.171526,07114
1,Jamaica Bay,2,Queens,40.610791,-73.822490,11693
2,Allerton/Pelham Gardens,3,Bronx,40.865745,-73.844947,10469
3,Alphabet City,4,Manhattan,40.724137,-73.977726,10009
4,Arden Heights,5,Staten Island,40.550665,-74.187537,10312


In [85]:
green_dfs.drop(['store_and_fwd_flag'], axis=1, inplace=True)

### Data Reformatting

In [86]:
# reformat zipcode to string
lookup_df['zipcode'] = lookup_df['zipcode'].astype(str)
lookup_df['borough'] = lookup_df['borough'].astype(str)
lookup_df.head()

,zone,LocationID,borough,latitude,longitude,zipcode
0,Newark Airport,1,EWR,40.689488,-74.171526,07114
1,Jamaica Bay,2,Queens,40.610791,-73.822490,11693
2,Allerton/Pelham Gardens,3,Bronx,40.865745,-73.844947,10469
3,Alphabet City,4,Manhattan,40.724137,-73.977726,10009
4,Arden Heights,5,Staten Island,40.550665,-74.187537,10312


In [87]:
green_dfs['lpep_pickup_datetime'] = pd.to_datetime(green_dfs['lpep_pickup_datetime'])
green_dfs['lpep_dropoff_datetime'] = pd.to_datetime(green_dfs['lpep_dropoff_datetime'])
green_dfs['passenger_count'] = pd.to_numeric(green_dfs['passenger_count'], errors='coerce').fillna(0).astype(int)
green_dfs['RatecodeID'] = pd.to_numeric(green_dfs['RatecodeID'], errors='coerce').fillna(99).astype(int)

### Data Transformation

In [88]:
lookup_df.head()

,zone,LocationID,borough,latitude,longitude,zipcode
0,Newark Airport,1,EWR,40.689488,-74.171526,07114
1,Jamaica Bay,2,Queens,40.610791,-73.822490,11693
2,Allerton/Pelham Gardens,3,Bronx,40.865745,-73.844947,10469
3,Alphabet City,4,Manhattan,40.724137,-73.977726,10009
4,Arden Heights,5,Staten Island,40.550665,-74.187537,10312


In [89]:
# join lookup and taxi_lookup based on locationid
lookup_df.rename(columns={'LocationID': 'location_id', 'latitude' : 'centroid_latitude', 

						   'longitude': 'centroid_longitude', 'zone':'zone_name' }, inplace=True)
lookup_df = lookup_df[['location_id', 'zone_name', 'borough', 'centroid_latitude', 'centroid_longitude']]
dim_location = lookup_df
dim_location.head()


,location_id,zone_name,borough,centroid_latitude,centroid_longitude
0,1,Newark Airport,EWR,40.689488,-74.171526
1,2,Jamaica Bay,Queens,40.610791,-73.822490
2,3,Allerton/Pelham Gardens,Bronx,40.865745,-73.844947
3,4,Alphabet City,Manhattan,40.724137,-73.977726
4,5,Arden Heights,Staten Island,40.550665,-74.187537


In [90]:
# Creating RateCode Dimension

# Mapping dictionary
ratecode_mapping = {
    1: 'Standard rate',
    2: 'JFK',
    3: 'Newark',
    4: 'Nassau or Westchester',
    5: 'Negotiated fare',
    6: 'Group ride',
	99: 'Unknown'
}

unique_ratecode_ids = green_dfs['RatecodeID'].unique()
# Converting the array of unique values into a DataFrame
dim_rate_code = pd.DataFrame(unique_ratecode_ids, columns=['rate_code_id'])

# Applying the mapping to create a new column with descriptions
dim_rate_code['rate_code_name'] = dim_rate_code['rate_code_id'].map(ratecode_mapping)
dim_rate_code = dim_rate_code[dim_rate_code['rate_code_id'] != '<NA>']
dim_rate_code

,rate_code_id,rate_code_name
0,1,Standard rate
1,5,Negotiated fare
2,4,Nassau or Westchester
3,3,Newark
4,2,JFK
5,99,Unknown
6,6,Group ride


In [91]:
# Creating Vendor Dimension

# Mapping dictionary
vendor_mapping = {
    1: 'Creative Mobile Technologies, LLC',
    2: 'Curb Mobility, LLC',
	6: 'Myle Technologies Inc'
}

unique_ids = green_dfs['VendorID'].unique()
# Converting the array of unique values into a DataFrame
dim_vendor = pd.DataFrame(unique_ids, columns=['vendor_id'])

# Applying the mapping to create a new column with descriptions
dim_vendor['vendor_name'] = dim_vendor['vendor_id'].map(vendor_mapping)
dim_vendor

,vendor_id,vendor_name
0,2,"Curb Mobility, LLC"
1,1,"Creative Mobile Technologies, LLC"


In [92]:
# Creating Trip Type Dimension

# Mapping dictionary
tripType_mapping = {
    1: 'Street-hail',
    2: 'Dispatch',
    3: 'Other',
    4: 'Other',
    5: 'Other',
}

unique_ids = green_dfs['trip_type'].unique()
# Converting the array of unique values into a DataFrame
dim_trip_type = pd.DataFrame(unique_ids, columns=['trip_type_id'])

# Applying the mapping to create a new column with descriptions
dim_trip_type['trip_type_name'] = dim_trip_type['trip_type_id'].map(tripType_mapping)
dim_trip_type = dim_trip_type[dim_trip_type['trip_type_id'] != '<NA>']
dim_trip_type = dim_trip_type.dropna(subset=['trip_type_id'])
dim_trip_type['trip_type_id'] = dim_trip_type['trip_type_id'].astype('Int64')
dim_trip_type

,trip_type_id,trip_type_name
0,1,Street-hail
1,2,Dispatch


In [93]:
# Creating Payment Type Dimension

# Mapping dictionary
payment_mapping = {
	0: 'Flex Fare trip',
    1: 'Credit card',
    2: 'Cash',
    3: 'No Charge',
    4: 'Dispute',
    5: 'Unknown',
    6: 'Voided trip'
}

unique_ids = green_dfs['payment_type'].unique()
# Converting the array of unique values into a DataFrame
dim_payment_type = pd.DataFrame(unique_ids, columns=['payment_type_id'])

# Applying the mapping to create a new column with descriptions
dim_payment_type['payment_type_name'] = dim_payment_type['payment_type_id'].map(payment_mapping)
dim_payment_type = dim_payment_type[dim_payment_type['payment_type_id'] != '<NA>']
dim_payment_type = dim_payment_type.dropna(subset=['payment_type_id'])
dim_payment_type['payment_type_id'] = dim_payment_type['payment_type_id'].astype('Int64')
dim_payment_type

,payment_type_id,payment_type_name
0,1,Credit card
1,2,Cash
2,3,No Charge
3,4,Dispute
4,5,Unknown


In [ ]:
green_dfs['pickup_date_id'] = green_dfs['lpep_pickup_datetime'].dt.strftime('%Y%m%d%H')
green_dfs['dropoff_date_id'] = green_dfs['lpep_dropoff_datetime'].dt.strftime('%Y%m%d%H')
green_dfs['trip_duration'] = (green_dfs['lpep_dropoff_datetime'] - green_dfs['lpep_pickup_datetime']).dt.total_seconds()
green_dfs.rename(columns={'VendorID': 'vendor_id', 'RatecodeID': 'rate_code_id', 'payment_type': 'payment_type_id'}, inplace=True)
green_dfs.rename(columns={'PULocationID': 'pickup_location_id', 'DOLocationID': 'dropoff_location_id'}, inplace=True)
green_dfs.rename(columns={'trip_type': 'trip_type_id'}, inplace=True)

green_dfs.drop(['lpep_pickup_datetime','lpep_dropoff_datetime'], axis=1, inplace=True)
green_dfs['payment_type_id'] = green_dfs['payment_type_id'].astype('Int64')
green_dfs['trip_type_id'] = green_dfs['trip_type_id'].astype('Int64')
green_dfs.head()

,vendor_id,rate_code_id,pickup_location_id,dropoff_location_id,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,...,ehail_fee,improvement_surcharge,total_amount,payment_type_id,trip_type_id,congestion_surcharge,cbd_congestion_fee,pickup_date_id,dropoff_date_id,trip_duration
0,2,1,236,239,1,1.98,12.8,1.0,0.5,3.61,...,NaN,1.0,21.66,1,1,2.75,NaN,2024010100,2024010100,690.0
1,2,1,65,170,5,6.54,30.3,1.0,0.5,7.11,...,NaN,1.0,42.66,1,1,2.75,NaN,2024010100,2024010100,1252.0
2,2,1,74,262,1,3.08,19.8,1.0,0.5,3.00,...,NaN,1.0,28.05,1,1,2.75,NaN,2024010100,2024010100,1142.0
3,1,1,74,116,1,2.40,14.2,1.0,1.5,0.00,...,NaN,1.0,16.70,2,1,0.00,NaN,2024010100,2024010100,712.0
4,2,1,74,243,1,5.14,22.6,1.0,0.5,6.28,...,NaN,1.0,31.38,1,1,0.00,NaN,2024010100,2024010100,659.0


### Data Consolidation

In [96]:
# Date Dimension

def week_of_month(dt):
    year = dt.year
    month = dt.month
    day = dt.day
    week_number = (day - 1) // 7 + 1
    return week_number

# calcualt the quarter
def get_quarter(month):
    if month in [1, 2, 3]:
        return 1
    elif month in [4, 5, 6]:
        return 2
    elif month in [7, 8, 9]:
        return 3
    else:
        return 4
    
# Calculate season
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"


start_date = pd.Timestamp("2022-01-01")
end_date = pd.Timestamp("2026-12-31")

dim_date = pd.DataFrame({'date': pd.date_range(start_date, end_date, freq='H')})

# Extract attributes
dim_date['year_number'] = dim_date['date'].dt.year
dim_date['month_number'] = dim_date['date'].dt.month
dim_date['day_number'] = dim_date['date'].dt.day
dim_date['month_name'] = dim_date['date'].dt.strftime('%B')
dim_date['day_name'] = dim_date['date'].dt.strftime('%A')
dim_date['hour_number'] = dim_date['date'].dt.hour
dim_date['timestamp_is_isoformat'] = dim_date['date'].apply(lambda x: x.isoformat())
dim_date['date_id'] = dim_date['date'].dt.strftime('%Y%m%d%H')
dim_date['week_of_month'] = dim_date['date'].apply(week_of_month)
dim_date['quarter_number'] = dim_date['month_number'].apply(get_quarter)
dim_date['week_of_year'] = dim_date['date'].dt.strftime('%U')
dim_date['season_name'] = dim_date['month_number'].apply(get_season)
dim_date['is_weekend'] = dim_date['day_name'].isin(['Saturday', 'Sunday'])

print(dim_date.shape)
print(dim_date.info())
dim_date.head()

(43801, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43801 entries, 0 to 43800
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   date                    43801 non-null  datetime64[ns]
 1   year_number             43801 non-null  int32         
 2   month_number            43801 non-null  int32         
 3   day_number              43801 non-null  int32         
 4   month_name              43801 non-null  object        
 5   day_name                43801 non-null  object        
 6   hour_number             43801 non-null  int32         
 7   timestamp_is_isoformat  43801 non-null  object        
 8   date_id                 43801 non-null  object        
 9   week_of_month           43801 non-null  int64         
 10  quarter_number          43801 non-null  int64         
 11  week_of_year            43801 non-null  object        
 12  season_name             43801 non-

,date,year_number,month_number,day_number,month_name,day_name,hour_number,timestamp_is_isoformat,date_id,week_of_month,quarter_number,week_of_year,season_name,is_weekend
0,2022-01-01 00:00:00,2022,1,1,January,Saturday,0,2022-01-01T00:00:00,2022010100,1,1,00,Winter,True
1,2022-01-01 01:00:00,2022,1,1,January,Saturday,1,2022-01-01T01:00:00,2022010101,1,1,00,Winter,True
2,2022-01-01 02:00:00,2022,1,1,January,Saturday,2,2022-01-01T02:00:00,2022010102,1,1,00,Winter,True
3,2022-01-01 03:00:00,2022,1,1,January,Saturday,3,2022-01-01T03:00:00,2022010103,1,1,00,Winter,True
4,2022-01-01 04:00:00,2022,1,1,January,Saturday,4,2022-01-01T04:00:00,2022010104,1,1,00,Winter,True


In [97]:
# combine both greendfs and yellowdfs
fact_trips = pd.concat([green_dfs], ignore_index=True)
print(fact_trips.shape)
print(fact_trips.info())
fact_trips.head()

(104877, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104877 entries, 0 to 104876
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   vendor_id              104877 non-null  int32  
 1   rate_code_id           104877 non-null  int64  
 2   pickup_location_id     104877 non-null  int32  
 3   dropoff_location_id    104877 non-null  int32  
 4   passenger_count        104877 non-null  int64  
 5   trip_distance          104877 non-null  float64
 6   fare_amount            104877 non-null  float64
 7   extra                  104877 non-null  float64
 8   mta_tax                104877 non-null  float64
 9   tip_amount             104877 non-null  float64
 10  tolls_amount           104877 non-null  float64
 11  ehail_fee              0 non-null       float64
 12  improvement_surcharge  104877 non-null  float64
 13  total_amount           104877 non-null  float64
 14  payment_type_id        

,vendor_id,rate_code_id,pickup_location_id,dropoff_location_id,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,...,ehail_fee,improvement_surcharge,total_amount,payment_type_id,trip_type_id,congestion_surcharge,cbd_congestion_fee,pickup_date_id,dropoff_date_id,trip_duration
0,2,1,236,239,1,1.98,12.8,1.0,0.5,3.61,...,NaN,1.0,21.66,1,1,2.75,NaN,2024010100,2024010100,690.0
1,2,1,65,170,5,6.54,30.3,1.0,0.5,7.11,...,NaN,1.0,42.66,1,1,2.75,NaN,2024010100,2024010100,1252.0
2,2,1,74,262,1,3.08,19.8,1.0,0.5,3.00,...,NaN,1.0,28.05,1,1,2.75,NaN,2024010100,2024010100,1142.0
3,1,1,74,116,1,2.40,14.2,1.0,1.5,0.00,...,NaN,1.0,16.70,2,1,0.00,NaN,2024010100,2024010100,712.0
4,2,1,74,243,1,5.14,22.6,1.0,0.5,6.28,...,NaN,1.0,31.38,1,1,0.00,NaN,2024010100,2024010100,659.0


## Loading

In [98]:
# loading dim_location
dim_location.to_sql('dim_location', engine, if_exists='append', index=False)

268

In [99]:
# loading dim_date
dim_date.to_sql('dim_date', engine, if_exists='append', index=False)

43801

In [100]:
dim_trip_type.to_sql('dim_trip_type', engine, if_exists='append', index=False)
dim_payment_type.to_sql('dim_payment_type', engine, if_exists='append', index=False)
dim_rate_code.to_sql('dim_rate_code', engine, if_exists='append', index=False)
dim_vendor.to_sql('dim_vendor', engine, if_exists='append', index=False)

2

In [102]:
fact_trips.shape

(104877, 21)

In [103]:
# Load Facts
chunksize = 50000
for i in range(0, fact_trips.shape[0], chunksize):
    chunk = fact_trips[i:i + chunksize]
    chunk.to_sql('fact_trips', engine, if_exists='append', index=False, method='multi')